In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
import warnings
warnings.filterwarnings('ignore')

In [5]:
df = pd.read_csv('dataset_filtré.csv')
print(f"Dataset shape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}\n")

Dataset shape: (2182, 13)
Missing values:
annee                      0
Item                       0
surface_cultivee_ha      922
production_tonnes          0
yield_value              911
rendement_tonnes_ha      911
pluviometrie_mm          730
temperature_moyenne_c    730
humidite_pct             730
ph                       126
azote_N                  126
phosphore_P_mg_kg        126
potassium_K_mg_kg        126
dtype: int64



In [3]:
df = df.dropna(subset=['rendement_tonnes_ha'])
print(f"Rows after dropping missing target: {len(df)}")

Rows after dropping missing target: 1271


In [4]:
df

,annee,Item,surface_cultivee_ha,production_tonnes,yield_value,rendement_tonnes_ha,pluviometrie_mm,temperature_moyenne_c,humidite_pct,ph,azote_N,phosphore_P_mg_kg,potassium_K_mg_kg
0,2010,"Almonds, in shell",162550.0,52000.00,319.9,0.03199,NaN,NaN,NaN,7.736923,0.110785,33.301538,228.518462
1,2010,"Anise, badian, coriander, cumin, caraway, fenn...",12965.0,10029.30,773.6,0.07736,NaN,NaN,NaN,7.736923,0.110785,33.301538,228.518462
2,2010,Apples,26294.0,126000.00,4791.9,0.47919,NaN,NaN,NaN,7.736923,0.110785,33.301538,228.518462
3,2010,Apricots,5982.0,23500.00,3928.7,0.39287,NaN,NaN,NaN,7.736923,0.110785,33.301538,228.518462
4,2010,Artichokes,1800.0,14000.00,7777.8,0.77778,NaN,NaN,NaN,7.736923,0.110785,33.301538,228.518462
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2177,2024,Unmanufactured tobacco,1584.0,2215.69,1398.5,0.13985,325.714286,20.414052,64.088603,NaN,NaN,NaN,NaN
2178,2024,Vegetables Primary,117743.0,3190802.74,27099.7,2.70997,325.714286,20.414052,64.088603,NaN,NaN,NaN,NaN
2179,2024,Vetches,102.0,254.13,2482.7,0.24827,325.714286,20.414052,64.088603,NaN,NaN,NaN,NaN
2180,2024,Watermelons,4648.0,504091.54,108448.2,10.84482,325.714286,20.414052,64.088603,NaN,NaN,NaN,NaN


In [5]:
# Encode crop type
le = LabelEncoder()
df['crop_encoded'] = le.fit_transform(df['Item'])

In [6]:
# NPK ratio features
df['NP_ratio'] = df['azote_N'] / (df['phosphore_P_mg_kg'] + 1e-5)
df['NK_ratio'] = df['azote_N'] / (df['potassium_K_mg_kg'] + 1e-5)
df['NPK_total'] = df['azote_N'] + df['phosphore_P_mg_kg'] + df['potassium_K_mg_kg']

In [7]:
df['temp_x_rain'] = df['temperature_moyenne_c'] * df['pluviometrie_mm']
df['log_surface'] = np.log1p(df['surface_cultivee_ha'])


In [8]:
# Features to use
FEATURES = [
    'crop_encoded', 'annee',
    'surface_cultivee_ha', 'log_surface',
    'pluviometrie_mm', 'temperature_moyenne_c', 'humidite_pct',
    'ph',
    'azote_N', 'phosphore_P_mg_kg', 'potassium_K_mg_kg',
    'NP_ratio', 'NK_ratio', 'NPK_total',
    'temp_x_rain'
]

In [9]:
TARGET = 'rendement_tonnes_ha'

X = df[FEATURES]
y = df[TARGET]

In [10]:
print(f"\nFeatures used: {FEATURES}")
print(f"Target: {TARGET}")
print(f"X shape: {X.shape}, y shape: {y.shape}\n")



Features used: ['crop_encoded', 'annee', 'surface_cultivee_ha', 'log_surface', 'pluviometrie_mm', 'temperature_moyenne_c', 'humidite_pct', 'ph', 'azote_N', 'phosphore_P_mg_kg', 'potassium_K_mg_kg', 'NP_ratio', 'NK_ratio', 'NPK_total', 'temp_x_rain']
Target: rendement_tonnes_ha
X shape: (1271, 15), y shape: (1271,)



In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
rf_pipeline = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

In [13]:
gb_pipeline = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(random_state=42))
])

In [14]:
print("🔍 GridSearchCV for Random Forest...")
param_grid_rf = {
    'model__n_estimators': [200, 400],
    'model__max_depth': [None, 20, 30],
    'model__min_samples_split': [2, 5],
    'model__max_features': ['sqrt', 'log2']
}


🔍 GridSearchCV for Random Forest...


In [15]:
grid_rf = GridSearchCV(
    rf_pipeline, param_grid_rf,
    cv=5, scoring='r2', n_jobs=-1, verbose=1
)
grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

Fitting 5 folds for each of 24 candidates, totalling 120 fits


In [16]:
print(f"\n✅ Best RF params: {grid_rf.best_params_}")
print(f"   CV R² (train): {grid_rf.best_score_:.4f}")



✅ Best RF params: {'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_split': 5, 'model__n_estimators': 200}
   CV R² (train): 0.4060


In [20]:
print("\n🔍 GridSearchCV for Gradient Boosting...")
param_grid_gb = {
    'model__n_estimators': [200, 400],
    'model__max_depth': [4, 6],
    'model__learning_rate': [0.05, 0.1],
    'model__subsample': [0.8, 1.0]
}


🔍 GridSearchCV for Gradient Boosting...


In [21]:
grid_gb = GridSearchCV(
    gb_pipeline, param_grid_gb,
    cv=5, scoring='r2', n_jobs=-1, verbose=1
)
grid_gb.fit(X_train, y_train)

best_gb = grid_gb.best_estimator_
print(f"\n✅ Best GB params: {grid_gb.best_params_}")
print(f"   CV R² (train): {grid_gb.best_score_:.4f}")


Fitting 5 folds for each of 16 candidates, totalling 80 fits

✅ Best GB params: {'model__learning_rate': 0.1, 'model__max_depth': 4, 'model__n_estimators': 400, 'model__subsample': 1.0}
   CV R² (train): 0.9573


In [22]:
print("\n🔗 Training Stacking Ensemble...")

# Build imputed arrays for stacking (sklearn stacking needs uniform pipeline)
imputer = KNNImputer(n_neighbors=5)
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=FEATURES)
X_test_imp  = pd.DataFrame(imputer.transform(X_test),  columns=FEATURES)


🔗 Training Stacking Ensemble...


In [23]:
rf_base  = RandomForestRegressor(**{k.replace('model__',''):v for k,v in grid_rf.best_params_.items()}, random_state=42, n_jobs=-1)
gb_base  = GradientBoostingRegressor(**{k.replace('model__',''):v for k,v in grid_gb.best_params_.items()}, random_state=42)


In [24]:
stacking = StackingRegressor(
    estimators=[('rf', rf_base), ('gb', gb_base)],
    final_estimator=Ridge(),
    cv=5,
    n_jobs=-1
)
stacking.fit(X_train_imp, y_train)
print("✅ Stacking model trained.")


✅ Stacking model trained.


In [29]:
def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    y_pred_train = model.predict(X_tr)
    y_pred_test  = model.predict(X_te)
    r2_tr  = r2_score(y_tr, y_pred_train)
    r2_te  = r2_score(y_te, y_pred_test)
    mae    = mean_absolute_error(y_te, y_pred_test)
    rmse   = np.sqrt(mean_squared_error(y_te, y_pred_test))
    print(f"\n{'─'*45}")
    print(f"  {name}")
    print(f"{'─'*45}")
    print(f"  R²  Train : {r2_tr:.4f}")
    print(f"  R²  Test  : {r2_te:.4f}  ← KEY METRIC")
    print(f"  MAE       : {mae:.4f} T/ha")
    print(f"  RMSE      : {rmse:.4f} T/ha")
    return r2_te

In [30]:
print("\n" + "═"*50)
print("       📊  MODEL EVALUATION RESULTS")
print("═"*50)
r2_rf  = evaluate("Random Forest (tuned)",      best_rf,  X_train, X_test, y_train, y_test)
r2_gb  = evaluate("Gradient Boosting (tuned)",  best_gb,  X_train, X_test, y_train, y_test)
r2_stk = evaluate("Stacking Ensemble (RF+GB→Ridge)", stacking, X_train_imp, X_test_imp, y_train, y_test)

print("\n" + "═"*50)
best_r2 = max(r2_rf, r2_gb, r2_stk)
best_name = ["Random Forest", "Gradient Boosting", "Stacking Ensemble"][np.argmax([r2_rf, r2_gb, r2_stk])]
print(f"  🏆 Best Model : {best_name}")
print(f"  🏆 Best R²    : {best_r2:.4f}")
print("═"*50)


══════════════════════════════════════════════════
       📊  MODEL EVALUATION RESULTS
══════════════════════════════════════════════════

─────────────────────────────────────────────
  Random Forest (tuned)
─────────────────────────────────────────────
  R²  Train : 0.8450
  R²  Test  : 0.4234  ← KEY METRIC
  MAE       : 0.7239 T/ha
  RMSE      : 1.0405 T/ha

─────────────────────────────────────────────
  Gradient Boosting (tuned)
─────────────────────────────────────────────
  R²  Train : 0.9992
  R²  Test  : 0.9515  ← KEY METRIC
  MAE       : 0.1109 T/ha
  RMSE      : 0.3019 T/ha

─────────────────────────────────────────────
  Stacking Ensemble (RF+GB→Ridge)
─────────────────────────────────────────────
  R²  Train : 0.9983
  R²  Test  : 0.9520  ← KEY METRIC
  MAE       : 0.1117 T/ha
  RMSE      : 0.3002 T/ha

══════════════════════════════════════════════════
  🏆 Best Model : Stacking Ensemble
  🏆 Best R²    : 0.9520
══════════════════════════════════════════════════


In [31]:
print("\n📌 Feature Importances (Random Forest):")
rf_model = best_rf.named_steps['model']
importances = pd.Series(rf_model.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=False)
for feat, imp in importances.items():
    bar = "█" * int(imp * 100)
    print(f"  {feat:<25} {imp:.4f}  {bar}")


📌 Feature Importances (Random Forest):
  crop_encoded              0.4539  █████████████████████████████████████████████
  surface_cultivee_ha       0.1992  ███████████████████
  log_surface               0.1907  ███████████████████
  temperature_moyenne_c     0.0205  ██
  humidite_pct              0.0161  █
  phosphore_P_mg_kg         0.0139  █
  annee                     0.0136  █
  pluviometrie_mm           0.0132  █
  NK_ratio                  0.0128  █
  temp_x_rain               0.0124  █
  azote_N                   0.0123  █
  NPK_total                 0.0112  █
  NP_ratio                  0.0111  █
  ph                        0.0098  
  potassium_K_mg_kg         0.0093  


In [32]:
print("\n🌾 Example Prediction:")
sample = X_test.iloc[[0]]
true_val = y_test.iloc[0]
pred_rf  = best_rf.predict(sample)[0]
pred_gb  = best_gb.predict(sample)[0]
print(f"  True Yield  : {true_val:.4f} T/ha")
print(f"  RF  Predict : {pred_rf:.4f} T/ha")
print(f"  GB  Predict : {pred_gb:.4f} T/ha")

print("\n✅ Done! Model ready for deployment.")



🌾 Example Prediction:
  True Yield  : 0.0710 T/ha
  RF  Predict : 1.3429 T/ha
  GB  Predict : 0.2912 T/ha

✅ Done! Model ready for deployment.
